# Fusion downstream — Stellar mass, conditioned on redshift

Slimmed-down mass probe keeping only the two best readouts, one per family:

- **im_spec_concat + XATTN** — the best *unaligned* method: cross-attention on the raw
  frozen-encoder image + spectrum patch tokens.
- **CLIP + MLP** — the best *aligned* method: an MLP on the frozen fusion (CLIP) latent.

Each method is trained **once on the full sample**, then predictions are split into
fixed-width redshift bins for the final grid (global training, per-redshift-bin evaluation).


In [ ]:
import sys, copy
from   tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
sys.path.insert(0, '.')
import h5py
from astropy.table import Table
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# == Config ====================================================================
H5       = "../data/crossmatched/embeddings_f150w.h5"   # DJA×F150W crossmatch (4 surveys)
DJA_FITS = "../data/spectrum/DJA_spectra_v4.5.fits"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

IMG_PATCH_KEY  = "image_patch_embed"    # (N, Ti, Di) image patch tokens
SPEC_KEY       = "spectrum_patch_embed" # (N, Ts, Ds) spectrum patch tokens
SMASK_KEY      = "spectrum_token_mask"  # (N, Ts) bool valid spectrum tokens
SPEC_STATS_KEY = "spectrum_stats"       # (N, 2)  raw f_lambda [mean, std] (absolute flux scale)
IMG_STATS_KEY  = "image_stats"          # (N, 2)  raw centre-crop [mean, std]

import glob as _glob, re as _re
CKPT = max(_glob.glob("outputs/fusion_dja_image_attn_spectrum_attn/version_*/checkpoints/*R10=*.ckpt"),
           key=lambda p: float(_re.search(r"R10=([0-9.]+)", p).group(1)))  # auto-pick best-R@10 checkpoint

LABEL_COL = "phot_mass"   # linear M*/Msun -> log10 (dex)
SN50_COL  = "sn50"
Z_COL     = "z_best"

# Sample filters (redshift is NOT cut here — it is the binning axis)
FRAC_VALID_PIX = 0.5      # keep spectra whose valid-token fraction > this
MIN_SN50       = 0
MIN_LOGM       = 7.5
MAX_LOGM       = 12.5

# Readout hyper-parameters
FS_LR      = 5e-4
HEAD_WD    = 1e-4
BATCH_SIZE = 256
D_MODEL    = 128          # cross-attention common dim
N_HEADS    = 4
HIDDEN     = 64
DROPOUT    = 0.04
SEED       = 42

# Downstream readout training budget
USE_EARLY_STOP = False   # True → early-stop on val MSE (BIN_PATIENCE); False → fixed step budget
MAX_STEPS      = 300    # total optimizer steps per z-bin when USE_EARLY_STOP=False (run to this, then stop)

# Redshift-bin evaluation
WIDTH_Z_BIN  = 0.5        # fixed bin width in z
BIN_VAL_FRAC = 0.50       # global val fraction (reported/plotted set)
BIN_MIN_N    = 10         # skip bins with fewer objects
BIN_EPOCHS   = 150
BIN_PATIENCE = 20
MAXZ_BIN     = 7.0        # drop the handful of very-high-z objects
UNALIGNED_HEAD = "xattn"  # best unaligned readout: im_spec_concat + cross-attention

print(f"device={DEVICE}  label={LABEL_COL}  z-bin width={WIDTH_Z_BIN}  unaligned head={UNALIGNED_HEAD}")

## Metrics, prediction plot, and the raw-token cross-attention head

In [ ]:
def metrics(y, yp):
    d = yp - y
    return dict(r2=r2_score(y, yp), mae=mean_absolute_error(y, yp),
                rmse=float(np.sqrt(np.mean(d ** 2))),
                nmad=1.4826 * np.median(np.abs(d - np.median(d))),
                out=float(np.mean(np.abs(d) > 0.5)))        # |dlogM| > 0.5 dex

def plot_pred(ax, y, yp, title, color=None):
    m = metrics(y, yp)
    lo, hi = np.floor(y.min()), np.ceil(y.max())
    ax.scatter(y, yp, s=6, alpha=0.4, color=color, rasterized=True)
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="1:1", alpha=0.4)
    xl = np.linspace(lo, hi, 100)
    ax.fill_between(xl, xl - 0.5, xl + 0.5, alpha=0.25, color="gray", label="+/-0.5 dex")
    ax.set_xlabel(r"$\log M_\star$ true"); ax.set_ylabel(r"$\log M_\star$ pred")
    ax.set_title(f"{title}\nR2={m['r2']:.3f}  NMAD={m['nmad']:.3f}  out={m['out']:.1%}", fontsize=10)
    ax.set_xlim(lo - 0.2, hi + 0.2); ax.set_ylim(lo - 0.2, hi + 0.2)
    ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
    return m

def _masked_mean(x, m):                 # torch: x (B,T,D), m (B,T) bool
    w = m.unsqueeze(-1).float()
    return (x * w).sum(1) / w.sum(1).clamp(min=1.0)


class MassHead(nn.Module):
    # im_spec_concat readout on raw tokens: image patch (B,Ti,Di) + spectrum patch (B,Ts,Ds) + (B,Ts) mask.
    def __init__(self, img_dim, spec_dim, head_kind="xattn", d_model=D_MODEL,
                 hidden=HIDDEN, dropout=DROPOUT, n_heads=N_HEADS):
        super().__init__()
        self.head_kind = head_kind
        def _stats_proj(out_dim):                # linear map of standardised [mean,std]
            m = nn.Linear(2, out_dim)
            nn.init.trunc_normal_(m.weight, std=0.02); nn.init.zeros_(m.bias)
            return m
        if head_kind in ("linear", "mlp"):
            self.img_ln  = nn.LayerNorm(img_dim)
            self.spec_ln = nn.LayerNorm(spec_dim)
            self.spec_stats_proj = _stats_proj(spec_dim)
            self.img_stats_proj  = _stats_proj(img_dim)
            in_dim = img_dim + spec_dim
            self.head = (nn.Linear(in_dim, 1) if head_kind == "linear" else
                         nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                       nn.Dropout(dropout), nn.Linear(hidden, 1)))
        elif head_kind == "xattn":
            self.spec_stats_proj = _stats_proj(d_model)
            self.img_stats_proj  = _stats_proj(d_model)
            self.img_proj  = nn.Sequential(nn.Linear(img_dim,  d_model), nn.LayerNorm(d_model))
            self.spec_proj = nn.Sequential(nn.Linear(spec_dim, d_model), nn.LayerNorm(d_model))
            self.reg  = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
            self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model)
            self.mlp  = nn.Sequential(nn.Linear(d_model, hidden), nn.ReLU(),
                                      nn.Dropout(dropout), nn.Linear(hidden, 1))
        else:
            raise ValueError(head_kind)

    def forward(self, img_tok, spec_tok, spec_mask, spec_stats, img_stats):
        B = img_tok.shape[0]
        if self.head_kind in ("linear", "mlp"):
            img_vec  = self.img_ln(img_tok.mean(1)) + self.img_stats_proj(img_stats)
            spec_vec = self.spec_ln(_masked_mean(spec_tok, spec_mask)) + self.spec_stats_proj(spec_stats)
            return self.head(torch.cat([img_vec, spec_vec], dim=-1))
        img  = self.img_proj(img_tok)                                # (B,Ti,D)
        spec = self.spec_proj(spec_tok)                              # (B,Ts,D)
        kv   = torch.cat([img, spec], dim=1)
        pad  = torch.cat([torch.zeros(B, img.shape[1], dtype=torch.bool, device=img.device),
                          ~spec_mask], dim=1)                        # True = ignore
        q = self.reg.expand(B, -1, -1)
        out, _ = self.attn(q, kv, kv, key_padding_mask=pad)
        vec = (self.norm(out.squeeze(1))
               + self.spec_stats_proj(spec_stats)
               + self.img_stats_proj(img_stats))
        return self.mlp(vec)

## Load the full sample (all filters except redshift) and the frozen CLIP latent

In [ ]:
from model.fusion import MultimodalFusion

dja = Table.read(DJA_FITS)

# --- load arrays: all config filters EXCEPT the redshift cut (full z range) ---
with h5py.File(H5, "r") as f:
    _ids = f["id"][:]
    _survey = f["survey"][:].astype(str)
    gIMGP = f[IMG_PATCH_KEY][:]; gSPEC = f[SPEC_KEY][:]; gSM = f[SMASK_KEY][:].astype(bool)
    gICLS = f["image_cls_embed"][:]                # DINO [CLS], for the unaligned baselines
    gSSR  = f[SPEC_STATS_KEY][:].astype(np.float32); gISR = f[IMG_STATS_KEY][:].astype(np.float32)
    _ai = np.isfinite(gIMGP.reshape(gIMGP.shape[0], -1)).all(1)
    _as = np.isfinite(gSPEC.reshape(gSPEC.shape[0], -1)).all(1)
    _vf = gSM.mean(1)
with np.errstate(invalid="ignore"):
    _logm = np.log10(np.asarray(dja[LABEL_COL])[_ids].astype(np.float32))
_sn = np.asarray(dja[SN50_COL])[_ids].astype(np.float32)
_z  = np.asarray(dja[Z_COL])[_ids].astype(np.float32)
_sel = _ai & _as & np.isfinite(_logm) & np.isfinite(_z) & (_z > 0)
if FRAC_VALID_PIX > 0:   _sel &= _vf > FRAC_VALID_PIX
if MIN_SN50 is not None: _sel &= _sn > MIN_SN50
if MIN_LOGM is not None: _sel &= _logm > MIN_LOGM
if MAX_LOGM is not None: _sel &= _logm <= MAX_LOGM
_sel &= _z <= MAXZ_BIN
_ix = np.where(_sel)[0]
gIMGP, gSPEC, gSM, gICLS = gIMGP[_ix], gSPEC[_ix], gSM[_ix], gICLS[_ix]
gSSR, gISR = gSSR[_ix], gISR[_ix]
gy, gz = _logm[_ix], _z[_ix]
gsurv  = _survey[_ix]
gdjaid = _ids[_ix]
print("sample: %d objects, z in [%.2f, %.2f], logM in [%.2f, %.2f]" %
      (len(_ix), gz.min(), gz.max(), gy.min(), gy.max()))

# --- build the frozen fusion (CLIP) head from the checkpoint ---
def build_fusion():
    ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    hp = ck["hyper_parameters"]
    in_dims = {"image": gIMGP.shape[-1], "spectrum": gSPEC.shape[-1]}
    fusion = MultimodalFusion(latent_dim=hp["latent_dim"], temperature=hp["temperature"])
    for name, cfg in hp["modalities"].items():
        fusion.register_modality(name, input_dim=in_dims[name], hidden_dim=cfg.get("hidden_dim"),
                                 pool=cfg.get("pool"), num_heads=cfg.get("num_heads", 4),
                                 stats_dim=cfg.get("stats_dim"))
    sd = {k[len("fusion."):]: v for k, v in ck["state_dict"].items() if k.startswith("fusion.")}
    fusion.load_state_dict(sd, strict=True)
    return fusion.to(DEVICE).eval(), hp

fusion, HP = build_fusion()
print("loaded CLIP head: latent_dim=%d  modalities=%s" % (fusion.latent_dim, list(fusion.projectors.keys())))

# --- frozen CLIP latent (image ++ spectrum) for the whole sample ---
@torch.no_grad()
def _clip_lat(bs=256):
    stats_mods = set(fusion.stats_encoders.keys()); _raw = {"spectrum": gSSR, "image": gISR}
    ei, es = [], []
    for s in range(0, len(_ix), bs):
        sl = slice(s, s + bs)
        im = torch.from_numpy(gIMGP[sl]).to(DEVICE); sp = torch.from_numpy(gSPEC[sl]).to(DEVICE)
        smk = torch.from_numpy(gSM[sl]).to(DEVICE)
        av = {n: torch.ones(im.shape[0], dtype=torch.bool, device=DEVICE) for n in ("image", "spectrum")}
        st = {m: torch.from_numpy(_raw[m][sl]).to(DEVICE) for m in stats_mods} or None
        emb = fusion({"image": im, "spectrum": sp}, av, {"spectrum": smk}, st)
        ei.append(emb["image"].cpu().numpy()); es.append(emb["spectrum"].cpu().numpy())
    return np.concatenate([np.concatenate(ei), np.concatenate(es)], -1)
gCLIP = _clip_lat()
print("clip latent: %s" % (gCLIP.shape,))

## The two readouts: im_spec_concat + XATTN (unaligned) and CLIP + MLP (aligned)

In [ ]:
def _iter(ix, bs, shuffle, rng):
    ix = np.asarray(ix)
    if shuffle: ix = rng.permutation(ix)
    for s in range(0, len(ix), bs):
        yield ix[s:s + bs]

# --- unaligned raw-token readout (MassHead, cross-attention) ---
def fit_unaligned(tr_i, va_i):
    rng = np.random.default_rng(SEED)
    S = np.arcsinh(gSSR); smu, ssd = S[tr_i].mean(0), S[tr_i].std(0) + 1e-6
    SST = ((S - smu) / ssd).astype(np.float32)
    I = np.arcsinh(gISR); imu, isd = np.nanmean(I[tr_i], 0), np.nanstd(I[tr_i], 0) + 1e-6
    IST = np.nan_to_num((I - imu) / isd, nan=0.0).astype(np.float32)
    ym, ysd = gy[tr_i].mean(), gy[tr_i].std() + 1e-6
    torch.manual_seed(SEED)
    model = MassHead(gIMGP.shape[-1], gSPEC.shape[-1], UNALIGNED_HEAD).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=FS_LR, weight_decay=HEAD_WD)
    def fwd(bi):
        return model(torch.from_numpy(gIMGP[bi]).to(DEVICE), torch.from_numpy(gSPEC[bi]).to(DEVICE),
                     torch.from_numpy(gSM[bi]).to(DEVICE), torch.from_numpy(SST[bi]).to(DEVICE),
                     torch.from_numpy(IST[bi]).to(DEVICE))
    def predict(ii):
        model.eval(); out = []
        with torch.no_grad():
            for bi in _iter(ii, 256, False, rng): out.append(fwd(bi).cpu().numpy())
        return np.concatenate(out).squeeze(1) * ysd + ym
    best, bstate, bad, step = 1e9, None, 0, 0
    for ep in range(BIN_EPOCHS if USE_EARLY_STOP else 10**9):
        model.train()
        for bi in _iter(tr_i, BATCH_SIZE, True, rng):
            yb = torch.from_numpy(((gy[bi] - ym) / ysd).astype(np.float32)).to(DEVICE).unsqueeze(1)
            opt.zero_grad(); loss = nn.functional.mse_loss(fwd(bi), yb); loss.backward(); opt.step()
            step += 1
            if not USE_EARLY_STOP and step >= MAX_STEPS: break
        if not USE_EARLY_STOP:                      # fixed step budget: no val, keep final model
            if step >= MAX_STEPS: break
            continue
        vp = predict(va_i); vl = float(np.mean((vp - gy[va_i]) ** 2))
        if vl < best - 1e-5: best, bstate, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= BIN_PATIENCE: break
    if USE_EARLY_STOP:
        model.load_state_dict(bstate)
    return predict(va_i)

# --- CLIP + MLP (MLP on the frozen fusion latent) ---
def fit_clipmlp(tr_i, va_i):
    ym, ysd = gy[tr_i].mean(), gy[tr_i].std() + 1e-6
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(gCLIP.shape[1], HIDDEN), nn.ReLU(),
                          nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 1)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=FS_LR, weight_decay=HEAD_WD)
    Xtr = torch.from_numpy(gCLIP[tr_i]).to(DEVICE)
    ytr = torch.from_numpy(((gy[tr_i] - ym) / ysd).astype(np.float32)).to(DEVICE).unsqueeze(1)
    Xva = torch.from_numpy(gCLIP[va_i]).to(DEVICE)
    yva = torch.from_numpy(((gy[va_i] - ym) / ysd).astype(np.float32)).to(DEVICE).unsqueeze(1)
    best, bstate, bad, step = 1e9, None, 0, 0
    for ep in range(BIN_EPOCHS if USE_EARLY_STOP else 10**9):
        model.train(); perm = torch.randperm(len(tr_i), device=DEVICE)
        for s in range(0, len(tr_i), BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]; opt.zero_grad()
            loss = nn.functional.mse_loss(model(Xtr[b]), ytr[b]); loss.backward(); opt.step()
            step += 1
            if not USE_EARLY_STOP and step >= MAX_STEPS: break
        if not USE_EARLY_STOP:                      # fixed step budget: no val, keep final model
            if step >= MAX_STEPS: break
            continue
        model.eval()
        with torch.no_grad(): vl = nn.functional.mse_loss(model(Xva), yva).item()
        if vl < best - 1e-5: best, bstate, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= BIN_PATIENCE: break
    if USE_EARLY_STOP:
        model.load_state_dict(bstate)
    model.eval()
    with torch.no_grad(): return model(Xva).cpu().numpy().squeeze(1) * ysd + ym

methods = [("im_spec_concat + %s" % UNALIGNED_HEAD.upper(), fit_unaligned, "royalblue"),
           ("CLIP + MLP", fit_clipmlp, "purple")]

# --- fixed-width redshift bins over the full range ---
edges = np.arange(0.0, np.ceil(gz.max() / WIDTH_Z_BIN) * WIDTH_Z_BIN + WIDTH_Z_BIN, WIDTH_Z_BIN)
counts = np.array([((gz >= edges[j]) & (gz < edges[j + 1])).sum() for j in range(len(edges) - 1)])
nz = np.where(counts > 0)[0]; edges = edges[nz[0]: nz[-1] + 2]; NB = len(edges) - 1
bcol = plt.cm.viridis(np.linspace(0.15, 0.85, NB))
print("z bins:", ["[%.1f,%.1f)" % (edges[j], edges[j+1]) for j in range(NB)])

## Global training, per-redshift-bin evaluation

In [ ]:
# one global train/val split; train each method once on the full sample
tr_all, va_all = train_test_split(np.arange(len(_ix)), test_size=BIN_VAL_FRAC,
                                  random_state=SEED, shuffle=True)
zva, yva = gz[va_all], gy[va_all]
survey_va = gsurv[va_all]
djaid_va  = gdjaid[va_all]
gpred = {}
for mi, (mname, fitfn, _) in tqdm(enumerate(methods), total=len(methods)):
    gpred[mi] = fitfn(tr_all, va_all)
    mm = metrics(yva, gpred[mi])
    print("GLOBAL %-22s n_tr=%d n_va=%d  R2=%.3f  sigmaNMAD=%.3f" %
          (mname, len(tr_all), len(va_all), mm["r2"], mm["nmad"]))

In [ ]:
# === Real + toy template spectra for the interpretability rows (cached in data/templates/) ===
# SF galaxy : EAZY v1.1 template set (Brammer+ 2008), sed2 = strong-emission-line star-forming.
# Quasar    : SDSS DR2 composite quasar template (Vanden Berk-style, 700-8500 A rest).
# SF (toy)  : the simple hand-built star-forming model (Balmer break + Gaussian emission lines).
# All are redshifted per z-bin and clipped to LowResPT's observed 1-2 um window, so each column
# shows which rest-frame features the spectrum encoder actually sees at that z.
from astropy.io import fits
TPL_DIR = "data/templates"

_sf = np.loadtxt(f"{TPL_DIR}/eazy_v1.1_sed2.dat")            # rest lambda [A], f_lambda
SF_W, SF_F = _sf[:, 0], _sf[:, 1]

_q  = fits.open(f"{TPL_DIR}/sdss_qso_composite.fit")          # SDSS composite quasar
_qh = _q[0].header; _qN = _q[0].data.shape[-1]
QSO_W = 10 ** (_qh["COEFF0"] + _qh["COEFF1"] * np.arange(_qN))  # rest lambda [A] (log grid)
QSO_F = _q[0].data[0]                                           # composite spectrum (row 0)

# rest-frame features to annotate (label, rest wavelength [A])
SF_LINES  = [("[OII]", 3727), (r"H$\beta$", 4861), ("[OIII]", 5007),
             (r"H$\alpha$", 6563), ("HeI", 10830), (r"Pa$\beta$", 12822)]
QSO_LINES = [(r"Ly$\alpha$", 1216), ("CIV", 1549), ("CIII]", 1909), ("MgII", 2798),
             ("[OII]", 3727), (r"H$\beta$", 4861), ("[OIII]", 5007), (r"H$\alpha$", 6563)]

WLO, WHI = 1.0, 2.0    # LowResPT observed-frame window [um] (wl_ref_min / wl_ref_max)
_ow = np.linspace(WLO, WHI, 800)

def tpl_in_window(w_rest_A, f, z, wlo=WLO, whi=WHI, npt=1200):
    """Redshift a rest-frame template and resample its flux onto [wlo, whi] um (NaN outside)."""
    obs_um = w_rest_A * (1.0 + z) / 1e4
    grid = np.linspace(wlo, whi, npt)
    return grid, np.interp(grid, obs_um, f, left=np.nan, right=np.nan)

# --- simple hand-built SF template: Balmer-break continuum + Gaussian emission lines ---
TOY_LINES = [("[OII]", 3727, 1.0), (r"H$\beta$", 4861, 0.7), ("[OIII]", 5007, 1.8),
             (r"H$\alpha$", 6563, 3.0), ("[SII]", 6724, 0.5), ("HeI", 10830, 0.8),
             (r"Pa$\beta$", 12822, 0.6), (r"Pa$\alpha$", 18756, 0.9)]  # (label, rest [A], strength)
def toy_sf_flux(z):
    """Toy SF spectrum on the observed 1-2 um grid: 4000A break + ~PRISM-width emission lines."""
    rest_um = _ow / (1.0 + z)
    flux = 0.55 + 0.45 / (1.0 + np.exp(-(rest_um - 0.4) / 0.008))            # 4000A Balmer break
    for _, lamA, amp in TOY_LINES:
        flux = flux + amp * np.exp(-0.5 * ((_ow - lamA * (1.0 + z) / 1e4) / 0.006) ** 2)
    return _ow, flux

print("templates: toy SF + EAZY sed2 SF (%.0f-%.0f A) + SDSS QSO composite (%.0f-%.0f A)"
      % (SF_W.min(), SF_W.max(), QSO_W.min(), QSO_W.max()))

## Poster figure — mass probe (both readouts) + real template rows across redshift bins

In [ ]:
# ============================================================
# POSTER FIGURE — stellar mass across redshift bins, GLOBAL training
#   TOP      : full-sample redshift distribution with the z-bins shaded
#   MIDDLE   : one row per method (im_spec_concat+XATTN, CLIP+MLP),
#              one square pred-vs-true panel per z-bin, titled (R2, sigmaNMAD)
#   BOTTOM   : two REAL template spectra (EAZY SF galaxy, SDSS composite quasar)
#              redshifted into the model's 1-2 um window per z-bin -> which rest-frame
#              features the spectrum encoder actually sees for each source type
#   LEADER LINES : z-bin edges (bottom of z-dist) -> top corners of the first method row
# ============================================================
import os
import matplotlib as mpl
from matplotlib.patches import ConnectionPatch

# Unified poster style (same POSTER_RC as st3 / recon / redshift notebooks)
POSTER_RC = {
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
}
mpl.rcParams.update(POSTER_RC)

POSTER_COLS = ["#e8000b", "#006bff"]     # im_spec_concat+XATTN (red), CLIP+MLP (blue)
NB_SHOW = NB - 1                         # drop the sparse last z-bin (poster convention)

# template rows (below the method rows): (label, flux_fn(z)->(grid_um, flux), line list [A], curve col, line col)
TPL_ROWS = [("SF (toy)\nmodel",   toy_sf_flux,                              TOY_LINES, "seagreen", "seagreen"),
            ("SF galaxy\n(EAZY)", lambda z: tpl_in_window(SF_W, SF_F, z),   SF_LINES,  "0.2",      "crimson"),
            ("Quasar\n(SDSS)",    lambda z: tpl_in_window(QSO_W, QSO_F, z), QSO_LINES, "#5a2d82",  "darkorange")]

def _bin_pred(mi, j):
    mask = (zva >= edges[j]) & (zva < edges[j + 1])
    if mask.sum() < BIN_MIN_N:
        return None
    return yva[mask], gpred[mi][mask]

# shared square limits across every shown pred-vs-true panel
_xy = [np.concatenate(d) for mi in range(len(methods)) for j in range(NB_SHOW)
       if (d := _bin_pred(mi, j)) is not None]
GLIM = (np.concatenate(_xy).min() - 0.2, np.concatenate(_xy).max() + 0.2) if _xy else None

# --- per-survey pred-vs-true rows (one method's predictions, split by survey) ---
SURVEY_MI = 0   # which method's predictions to break down by survey (0 = im_spec_concat+XATTN)
SURVEY_PALETTE = {"cosmos": "#0072B2", "ceers": "#E69F00", "jades": "#CC79A7", "outthere": "#009E73"}
SURVEY_ROWS = [s for s in ["cosmos", "ceers", "jades", "outthere"] if np.any(survey_va == s)]

def _bin_pred_survey(s, j):
    mask = (zva >= edges[j]) & (zva < edges[j + 1]) & (survey_va == s)
    if mask.sum() < BIN_MIN_N:
        return None
    return yva[mask], gpred[SURVEY_MI][mask]

SPEC_N_ROWS = 3    # bottom rows: random original val spectra per z-bin
NROW = len(methods) + len(SURVEY_ROWS) + len(TPL_ROWS) + SPEC_N_ROWS
fig = plt.figure(figsize=(2.8 * NB_SHOW,
                          3.0 * len(methods) + 3.0 * len(SURVEY_ROWS) + 1.8
                          + 2.0 * len(TPL_ROWS) + 2.0 * SPEC_N_ROWS))
gs  = gridspec.GridSpec(NROW + 1, NB_SHOW,
                        height_ratios=[0.28] + [1.0] * len(methods)
                                      + [1.0] * len(SURVEY_ROWS) + [0.8] * len(TPL_ROWS)
                                      + [0.8] * SPEC_N_ROWS,
                        hspace=0.42, wspace=0.32)

# --- TOP: redshift distribution + shaded bins ---
axz = fig.add_subplot(gs[0, :])
axz.hist(gz, bins=max(40, NB * 3), color="0.75", edgecolor="white")
for j in range(NB_SHOW):
    axz.axvspan(edges[j], edges[j + 1], color=bcol[j], alpha=0.12)
for e in edges:
    axz.axvline(e, color="k", lw=0.7, ls="--", alpha=0.45)
axz.set_ylabel("count")
axz.set_xlim(edges[0] - 0.2, edges[NB_SHOW] + 0.2)
axz.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))
axz.tick_params(axis='both', labelsize=14)
axz.set_title("Validation-sample redshift distribution", fontsize=13, fontweight="bold", pad=6)

# --- method rows: per-bin pred-vs-true panels ---
top_panels = []
for mi, (mname, _, _) in enumerate(methods):
    col = POSTER_COLS[mi % len(POSTER_COLS)]
    for j in range(NB_SHOW):
        ax = fig.add_subplot(gs[mi + 1, j])
        ax.set_box_aspect(1); ax.set_anchor('N')
        if mi == 0: top_panels.append(ax)
        d = _bin_pred(mi, j)
        if d is not None:
            yv, pv = d
            m = plot_pred(ax, yv, pv, "", col)
            (a_fit, b_fit), _cov = np.polyfit(yv, pv, 1, cov=True)    # linear fit: pred = a*true + b
            a_err, b_err = np.sqrt(np.diag(_cov))                     # 1-sigma errors on slope / intercept
            _xf = np.array(GLIM) if GLIM is not None else np.array([yv.min(), yv.max()])
            ax.plot(_xf, a_fit * _xf + b_fit, color="k", lw=1.4, alpha=0.85, zorder=5)   # fit line
            if not (mi == 0 and j == 0) and ax.get_legend() is not None:
                ax.get_legend().remove()
            ax.set_title(rf"$R^2$={m['r2']:.2f}; $\sigma_{{\rm NMAD}}$={m['nmad']:.2f}", fontsize=11, pad=2)
            ax.text(0.96, 0.05,
                    f"slope={a_fit:.2f}$\\pm${a_err:.2f}\n$b$={b_fit:+.2f}$\\pm${b_err:.2f}\nn={len(yv)}",
                    transform=ax.transAxes, ha='right', va='bottom', fontsize=8)
            ax.set_xlabel(r"$\log M_\star$ true", fontsize=10, labelpad=1)
            if GLIM is not None: ax.set_xlim(*GLIM); ax.set_ylim(*GLIM)
            ax.tick_params(axis='both', labelsize=12)
        else:
            ax.set_title("z [%.2f, %.2f)  (too few)" % (edges[j], edges[j + 1]), fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(mname + "\n" + r"$\log M_\star$ pred", fontsize=11)
        else:
            ax.set_ylabel(None)

# --- per-survey rows: method SURVEY_MI's pred-vs-true, split by survey ---
for si, s in enumerate(SURVEY_ROWS):
    scol = SURVEY_PALETTE.get(s, "0.3")
    for j in range(NB_SHOW):
        ax = fig.add_subplot(gs[len(methods) + 1 + si, j])
        ax.set_box_aspect(1); ax.set_anchor('N')
        d = _bin_pred_survey(s, j)
        if d is not None:
            yv, pv = d
            m = plot_pred(ax, yv, pv, "", scol)
            if ax.get_legend() is not None:
                ax.get_legend().remove()
            ax.set_title(rf"$R^2$={m['r2']:.2f}; $\sigma_{{\rm NMAD}}$={m['nmad']:.2f}", fontsize=11, pad=2)
            ax.text(0.96, 0.05, f"n={len(yv)}", transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=8)
            ax.set_xlabel(r"$\log M_\star$ true", fontsize=10, labelpad=1)
            if GLIM is not None:
                ax.set_xlim(*GLIM); ax.set_ylim(*GLIM)
            ax.tick_params(axis='both', labelsize=12)
        else:
            ax.set_title("z [%.2f, %.2f)  (too few)" % (edges[j], edges[j + 1]), fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"{s}\n" + r"$\log M_\star$ pred", fontsize=11)
        else:
            ax.set_ylabel(None)

# --- BOTTOM: real SF-galaxy & quasar templates redshifted into the 1-2 um window ---
# Each column redshifts the template to its z-bin centre; vertical lines mark the
# rest-frame features that land in-band (per-panel flux normalised for display).
for ri, (rname, flux_fn, LINES, ccol, lcol) in enumerate(TPL_ROWS):
    row = len(methods) + len(SURVEY_ROWS) + 1 + ri
    for j in range(NB_SHOW):
        ax = fig.add_subplot(gs[row, j])
        zc = 0.5 * (edges[j] + edges[j + 1])
        g, fl = flux_fn(zc)
        if np.isfinite(fl).any():
            ax.plot(g, fl / np.nanmax(fl), color=ccol, lw=1.2)
        ax.set_xlim(WLO, WHI); ax.set_ylim(0, 1.2); ax.set_yticks([])
        for label, lam0 in ((ln[0], ln[1]) for ln in LINES):   # tolerate stale 3-tuple line lists
            obs = lam0 * (1.0 + zc) / 1e4
            if WLO < obs < WHI:
                ax.axvline(obs, color=lcol, lw=0.8, ls="--", alpha=0.75)
                ax.text(obs, 1.16, label, rotation=90, ha="right", va="top",
                        fontsize=7, color=lcol)
        if ri == 0:
            ax.set_title(r"$z\approx%.1f$" % zc, fontsize=11, pad=2)
        if ri == len(TPL_ROWS) - 1:
            ax.set_xlabel(r"obs $\lambda$ [$\mu$m]", fontsize=10, labelpad=1)
        else:
            ax.set_xticklabels([])
        ax.tick_params(axis='x', labelsize=9)
        if j == 0:
            ax.set_ylabel(rname + "\n" + r"$f_\lambda$", fontsize=10)

# --- BOTTOM ROWS: SPEC_N_ROWS random original val spectra per z-bin ---
from astropy.io import fits as _fits
_wave = np.asarray(_fits.open(DJA_FITS)["WAVE"].data, dtype=float)   # (473,) observed um
_flux_all  = np.asarray(dja["flux"])                                  # (Ndja, 473) raw flux
_vspec_all = np.asarray(dja["valid_spec"]).astype(bool)
_spec_rng  = np.random.default_rng(SEED)
_bin_picks = {}
for j in range(NB_SHOW):
    pool = np.where((zva >= edges[j]) & (zva < edges[j + 1]))[0]
    _bin_picks[j] = (_spec_rng.choice(pool, size=min(SPEC_N_ROWS, len(pool)), replace=False)
                     if len(pool) else np.array([], int))
_spec_row0 = len(methods) + len(SURVEY_ROWS) + len(TPL_ROWS) + 1
for r in range(SPEC_N_ROWS):
    for j in range(NB_SHOW):
        ax = fig.add_subplot(gs[_spec_row0 + r, j])
        picks = _bin_picks[j]
        if r < len(picks):
            vi  = int(picks[r]); did = int(djaid_va[vi])
            fl  = _flux_all[did].astype(float).copy()
            fl[~(_vspec_all[did] & np.isfinite(fl))] = np.nan
            mw = (_wave >= WLO) & (_wave <= WHI)
            ax.plot(_wave[mw], fl[mw], color="0.25", lw=0.8)
            ax.set_xlim(WLO, WHI); ax.set_yticks([])
            ax.set_title(f"z={zva[vi]:.2f}", fontsize=8, pad=2)
        else:
            ax.set_xticks([]); ax.set_yticks([])
        if r == SPEC_N_ROWS - 1:
            ax.set_xlabel(r"obs $\lambda$ [$\mu$m]", fontsize=9, labelpad=1)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(f"val spec {r + 1}\n" + r"$f$", fontsize=9)

fig.tight_layout()

# --- LEADER LINES: bin edges (bottom of z-dist) -> top corners of the first method row ---
for j, ax in enumerate(top_panels):
    for x_edge, x_corner in ((edges[j], 0.0), (edges[j + 1], 1.0)):
        con = ConnectionPatch(
            xyA=(x_edge, 0.0), coordsA=axz.get_xaxis_transform(),
            xyB=(x_corner, 1.0), coordsB=ax.transAxes,
            color=bcol[j], lw=1.6, alpha=0.75, zorder=0)
        con.set_clip_on(False); fig.add_artist(con)

POSTER_DIR = "/home/yacheng/ssl_outthere/poster_figs"
os.makedirs(POSTER_DIR, exist_ok=True)
outp = os.path.join(POSTER_DIR, "poster_mass_conditioned_zbins")
fig.savefig(outp + ".png", transparent=True)
print(f"Saved -> {outp}.png")
plt.show()

# --- per (method, z-bin) metrics table ---
print("\n%-22s %-16s %5s %8s %8s" % ("method", "z-bin", "n", "R2", "NMAD"))
print("-" * 62)
for mi, (mname, _, _) in enumerate(methods):
    for j in range(NB_SHOW):
        d = _bin_pred(mi, j)
        zb = "[%.2f,%.2f)" % (edges[j], edges[j + 1])
        if d is None:
            print("%-22s %-16s %5s" % (mname, zb, "(too few)")); continue
        mm = metrics(*d)
        print("%-22s %-16s %5d %8.3f %8.3f" % (mname, zb, len(d[0]), mm["r2"], mm["nmad"]))

## Linear probe (closed-form ridge) on the frozen CLIP latent

Same sample, same global 50/50 split, same z-bin grid as the figure above, but the
readout is a closed-form ridge regression instead of a trained MLP head -- no
optimizer, no step budget. Reuses `edges2`, `bcol2`, `GLIM2`, `_bin_mask2` and the
survey palette defined in the figure cell above.


In [ ]:
# ridge linear probe on the frozen CLIP latent, plotted on the same z-bin grid
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

RIDGE_ALPHAS = np.logspace(-2, 6, 25)
RIDGE_ROWS = [("CLIP + ridge", gCLIP)]        # add (name, feature matrix) to stack more rows

def fit_ridge(X, tr_i, va_i):
    sc  = StandardScaler().fit(X[tr_i])
    reg = RidgeCV(alphas=RIDGE_ALPHAS).fit(sc.transform(X[tr_i]), gy[tr_i])
    return reg.predict(sc.transform(X[va_i])), reg.alpha_

rpred = {}
for ri, (rname, X) in enumerate(RIDGE_ROWS):
    rpred[ri], a = fit_ridge(X, tr_all, va_all)
    mm = metrics(yva, rpred[ri])
    print("RIDGE %-16s alpha=%-8.3g R2=%.3f  sigmaNMAD=%.3f  out=%.1f%%"
          % (rname, a, mm["r2"], mm["nmad"], 100 * mm["out"]))

fig = plt.figure(figsize=(15, 4.4 * len(RIDGE_ROWS)))
outer = gridspec.GridSpec(2, 1, height_ratios=[0.20, 1.0 * len(RIDGE_ROWS)], hspace=0.10)
gs = outer[1].subgridspec(len(RIDGE_ROWS), NB2, hspace=0.05, wspace=0.03)

axz = fig.add_subplot(outer[0])
zbins = np.linspace(edges2[0], edges2[-1], max(40, NB2 * 8) + 1)
axz.hist(gz, bins=zbins, color="0.75", edgecolor="white", label="all")
axz.hist(gz[tr_all], bins=zbins, histtype="step", color="k", lw=1.6, label="train")
for j in range(NB2):
    axz.axvspan(edges2[j], edges2[j + 1], color=bcol2[j], alpha=0.22)
for e in edges2:
    axz.axvline(e, color="k", lw=1.0, ls="--", alpha=0.7)
axz.set_ylabel("count", fontsize=12, labelpad=2)
axz.yaxis.set_major_locator(mpl.ticker.MaxNLocator(2, prune="lower"))
axz.set_xlim(edges2[0], edges2[-1])
for sp in axz.spines.values():
    sp.set_linewidth(1.8)
axz.tick_params(axis='both', which='both', direction='in', width=1.6, length=4.2,
                labelsize=11, top=True, bottom=True, left=True, right=True)
axz.xaxis.set_ticks_position('top'); axz.xaxis.set_label_position('top')
axz.tick_params(axis='x', labeltop=True, labelbottom=False, labelsize=13)
axz.set_xlabel("redshift $z$", fontsize=12, labelpad=2)
axz.legend(fontsize=11, loc="upper right", frameon=False, handlelength=1.2,
           handletextpad=0.4, borderaxespad=0.2, bbox_to_anchor=(1.0, 0.98))

top_panels = []
for ri, (rname, _) in enumerate(RIDGE_ROWS):
    row_first, bottom = None, ri == len(RIDGE_ROWS) - 1
    for j in range(NB2):
        ax = fig.add_subplot(gs[ri, j], sharey=row_first)
        if row_first is None:
            row_first = ax
        ax.set_box_aspect(1); ax.grid(False)
        if ri == 0:
            top_panels.append(ax)
        mk = _bin_mask2(j)
        if mk is not None:
            yv, pv, sv = yva[mk], rpred[ri][mk], survey_va[mk]
            for s in _svorder:
                m2 = sv == s
                if m2.any():
                    ax.scatter(yv[m2], pv[m2], s=13, alpha=0.3, color=SURVEY_PALETTE[s],
                               edgecolors="none", rasterized=True,
                               label=s if (ri == 0 and j == 0) else None)
            ax.plot(GLIM2, GLIM2, "--", color="0.35", lw=1.2, alpha=0.75, dashes=(5, 3))
            mm = metrics(yv, pv)
            ax.text(0.96, 0.05,
                    rf"$R^2$={mm['r2']:.2f}" + "\n" + rf"$\sigma_{{\rm NMAD}}$={mm['nmad']:.2f}",
                    transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
                    bbox=dict(facecolor='white', alpha=0.7, pad=2.0, edgecolor='none'))
            if ri == 0:
                ax.text(0.05, 0.95, f"n={len(yv)}", transform=ax.transAxes, ha='left',
                        va='top', fontsize=11,
                        bbox=dict(facecolor='white', alpha=0.7, pad=2.0, edgecolor='none'))
        else:
            ax.text(0.5, 0.5, f"n<{BIN_MIN_N}", transform=ax.transAxes, ha='center',
                    va='center', fontsize=11, color='0.5')
        ax.set_xlim(*GLIM2); ax.set_ylim(*GLIM2)
        ax.set_xticks(TICKS); ax.set_yticks(TICKS)
        for sp in ax.spines.values():
            sp.set_linewidth(1.8)
        ax.tick_params(which='both', direction='in', width=1.6, length=4.2, labelsize=11,
                       top=True, bottom=True, left=True, right=True)
        if j == 0:
            ax.set_ylabel(r"$\log M_\star$ pred", fontsize=12)
        else:
            ax.tick_params(labelleft=False)
        if bottom:
            ax.set_xlabel(r"$\log M_\star$ true", fontsize=12)
        else:
            ax.tick_params(labelbottom=False)

fig.tight_layout()
handles, labels = top_panels[0].get_legend_handles_labels()
leg = top_panels[-1].legend(handles, labels, loc="upper right", ncol=1, frameon=True,
                            markerscale=2.2, handletextpad=0.35, labelspacing=0.3,
                            borderpad=0.3, prop={'size': 11}, bbox_to_anchor=(0.98, 0.98))
leg.get_frame().set_edgecolor('none'); leg.get_frame().set_alpha(0.7); leg.set_zorder(5)

for j, ax in enumerate(top_panels):
    for x_edge, x_corner in ((edges2[j], 0.0), (edges2[j + 1], 1.0)):
        con = ConnectionPatch(xyA=(x_edge, 0.0), coordsA=axz.get_xaxis_transform(),
                              xyB=(x_corner, 1.0), coordsB=ax.transAxes,
                              color=bcol2[j], lw=2.4, alpha=1.0, zorder=0)
        con.set_clip_on(False); fig.add_artist(con)

fig.patch.set_alpha(0.0)
outp = os.path.join(POSTER_DIR, 'mass_conditioned_redshift_ridge')
fig.savefig(outp + '.png', transparent=False, dpi=300, bbox_inches='tight')
fig.savefig('/home/yacheng/nexus/ssl_outthere/paper/ssl_outthere_paper/mass.png', transparent=False, dpi=500, bbox_inches='tight')
print(f'Saved -> {outp}.png')
plt.show()


## Table: aligned vs unaligned representations at $1<z\le3$

Reproduces the paper's stellar-mass table. The unaligned readouts pool the frozen encoder
tokens with **no learned parameters**, and are kept close in width to the 256-d aligned
embedding so the comparison is not confounded by dimensionality: image = `[CLS]` + mean
patch (1026-d), spectrum = masked mean + masked max (258-d); both append the raw
$(\mu,\sigma)$ flux statistics. Wider and narrower pooling variants are listed alongside
the paper rows (marked `*`) so the sensitivity to that choice is visible in the same table.
Everything is trained and tested inside $1<z\le3$, averaged over `N_SPLITS` random
$50\%/50\%$ splits, with the aligned/unaligned comparison paired within each split.


In [ ]:
# === 1 < z <= 3: photometry baseline vs (un)aligned representations, ridge readout ===
# Paper Table 1. The unaligned readouts pool the frozen tokens with no learned parameters and
# are kept close in width to the 256-d aligned embedding, so the comparison is not confounded
# by dimensionality: image = CLS + mean patch, spectrum = masked mean + masked max. Both
# append the raw (mean, std) flux statistics. Wider/narrower variants are kept below the
# paper rows so the sensitivity to the pooling choice can be read off the same table.
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

ZLO, ZHI  = 1.0, 3.0
N_SPLITS  = 40
ALPHAS    = np.logspace(-2, 6, 25)

_f150 = np.asarray(dja["phot_f150w_tot_1"])[gdjaid].astype(np.float64)
zmask = (gz > ZLO) & (gz <= ZHI) & np.isfinite(_f150) & (_f150 > 0)
zi    = np.where(zmask)[0]

_m      = gSM[zi]                                  # (n, T) bool valid spectrum tokens
_w      = _m[..., None].astype(np.float32)
_ist    = np.arcsinh(gISR[zi])
_sst    = np.arcsinh(gSSR[zi])
_imean  = gIMGP[zi].mean(1)
_smean  = (gSPEC[zi] * _w).sum(1) / np.clip(_w.sum(1), 1, None)
# masked max over valid tokens only (matches MaskedMaxPool in model/fusion.py)
_smax   = np.nan_to_num(np.where(_m[..., None], gSPEC[zi], -np.inf).max(1), neginf=0.0)

PHOT     = np.log10(_f150[zi])[:, None]
IMG_MEAN = np.hstack([_imean, _ist])                                   # patch mean-pool
IMG_CLS  = np.hstack([gICLS[zi], _ist])                                # DINO CLS token
IMG_BASE = np.hstack([gICLS[zi], _imean, _ist])                        # * CLS + mean patch
SPC_MEAN = np.hstack([_smean, _sst])                                   # masked mean-pool
SPC_BASE = np.hstack([_smean, _smax, _sst])                            # * masked mean + max
SPC_CAT  = np.hstack([(gSPEC[zi] * _w).reshape(len(zi), -1), _sst])    # all patch tokens
L        = gCLIP.shape[1] // 2
IMG_AL, SPC_AL = gCLIP[zi, :L], gCLIP[zi, L:]
yz       = gy[zi]

READOUTS = [
    ("F150W photometry  *",             PHOT),
    ("image: patch mean-pool",          IMG_MEAN),
    ("image: CLS token",                IMG_CLS),
    ("image: CLS + mean patch  *",      IMG_BASE),
    ("image: CLIP aligned  *",          IMG_AL),
    ("spectrum: masked mean-pool",      SPC_MEAN),
    ("spectrum: mean + max  *",         SPC_BASE),
    ("spectrum: all patch tokens",      SPC_CAT),
    ("spectrum: CLIP aligned  *",       SPC_AL),
    ("joint: mean-pool both",           np.hstack([IMG_MEAN, SPC_MEAN])),
    ("joint: unaligned  *",             np.hstack([IMG_BASE, SPC_BASE])),
    ("joint: all tokens",               np.hstack([IMG_BASE, SPC_CAT])),
    ("joint: CLIP aligned  *",          gCLIP[zi]),
]

def ridge_probe(X, tr, va):
    sc = StandardScaler().fit(X[tr])
    p  = RidgeCV(alphas=ALPHAS).fit(sc.transform(X[tr]), yz[tr]).predict(sc.transform(X[va]))
    d  = p - yz[va]
    return (r2_score(yz[va], p),
            1.4826 * np.median(np.abs(d - np.median(d))),
            float(np.mean(np.abs(d) > 0.5)))

splits = [train_test_split(np.arange(len(zi)), test_size=0.5, random_state=s)
          for s in range(N_SPLITS)]
print(f"1 < z <= 3: n={len(zi)} objects ({len(np.unique(gdjaid[zi]))} unique), "
      f"{N_SPLITS} random 50/50 splits;  * = row quoted in the paper\n")
hdr = (f"{'readout':30s} {'dim':>6s} {'R2':>16s} {'sigma_NMAD':>16s} {'f>0.5':>15s}")
print(hdr); print("-" * len(hdr))
RES = {}
for name, X in READOUTS:
    R = np.array([ridge_probe(X, tr, va) for tr, va in splits])
    RES[name] = R
    print(f"{name:30s} {X.shape[1]:6d} "
          f"{R[:,0].mean():8.3f}+/-{R[:,0].std():5.3f} "
          f"{R[:,1].mean():8.3f}+/-{R[:,1].std():5.3f} "
          f"{100*R[:,2].mean():6.1f}+/-{100*R[:,2].std():4.1f}%")

print("\npaired effect of alignment on the paper rows (same splits):")
for al, un in (("image: CLIP aligned  *",    "image: CLS + mean patch  *"),
               ("spectrum: CLIP aligned  *", "spectrum: mean + max  *"),
               ("joint: CLIP aligned  *",    "joint: unaligned  *")):
    for j, lab in ((0, "dR2  "), (1, "dNMAD")):
        d = RES[al][:, j] - RES[un][:, j]
        print(f"  {un:28s} -> aligned  {lab} = {d.mean():+.4f} +/- {d.std(ddof=1):.4f} "
              f"({abs(d.mean() / d.std(ddof=1)):.1f} sigma)")

print("\ncommonality decomposition of log M* (shared = RI + RS - RJ):")
for tag, ki, ks, kj in (("un.", "image: CLS + mean patch  *", "spectrum: mean + max  *",
                         "joint: unaligned  *"),
                        ("al.", "image: CLIP aligned  *", "spectrum: CLIP aligned  *",
                         "joint: CLIP aligned  *")):
    rI, rS, rJ = RES[ki][:, 0], RES[ks][:, 0], RES[kj][:, 0]
    sh, ui, us = rI + rS - rJ, rJ - rS, rJ - rI
    print(f"  {tag}  shared={sh.mean():.3f}+/-{sh.std(ddof=1):.3f}  "
          f"uniq_img={ui.mean():.3f}+/-{ui.std(ddof=1):.3f}  "
          f"uniq_spec={us.mean():.3f}+/-{us.std(ddof=1):.3f}")


### Control: is the gain just a better pooling?

The aligned embedding is produced by a learned attention pooling, the unaligned readouts by
fixed pooling, so the comparison above confounds *alignment* with *learned pooling*. Here the
same attention-pooling architecture is trained **directly on $\log M_\star$** on the frozen
tokens --- with access to the labels the CLIP head never sees. If learned pooling were the
explanation, this supervised head should match the aligned embedding.


In [ ]:
# === supervised attention pooling + linear readout, per modality (pooling control) ===
import torch.nn as nn

D_MODEL, N_HEADS, EPOCHS, LR_H, WD_H, BS_H, PAT_H = 128, 4, 300, 1e-3, 1e-4, 128, 30
CTRL_SPLITS = 5                       # this one trains a network per split, so keep it small

class AttnPoolHead(nn.Module):
    """Learnable-query attention pooling over frozen tokens, plus stats, plus a linear head."""
    def __init__(self, d_in, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.proj  = nn.Sequential(nn.Linear(d_in, d_model), nn.LayerNorm(d_model))
        self.stats = nn.Linear(2, d_model)
        self.q     = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.out   = nn.Linear(d_model, 1)

    def forward(self, tok, stats, mask=None):
        h = self.proj(tok)
        v, _ = self.attn(self.q.expand(h.shape[0], -1, -1), h, h,
                         key_padding_mask=None if mask is None else ~mask)
        return self.out(self.norm(v.squeeze(1)) + self.stats(stats))

def train_attnpool(tok, stats, mask, tr, va, seed=0):
    torch.manual_seed(seed)
    ym, ysd = yz[tr].mean(), yz[tr].std()
    inner_tr, inner_va = train_test_split(tr, test_size=0.15, random_state=seed)
    T = torch.from_numpy(tok).float().to(DEVICE)
    S = torch.from_numpy(((stats - stats[tr].mean(0)) / (stats[tr].std(0) + 1e-6)).astype(np.float32)).to(DEVICE)
    M = None if mask is None else torch.from_numpy(mask).to(DEVICE)
    Y = torch.from_numpy(((yz - ym) / ysd).astype(np.float32)).to(DEVICE).unsqueeze(1)
    model = AttnPoolHead(tok.shape[-1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_H, weight_decay=WD_H)
    best, bstate, bad = 1e9, None, 0
    for ep in range(EPOCHS):
        model.train()
        perm = np.random.default_rng(seed * 1000 + ep).permutation(inner_tr)
        for s in range(0, len(perm), BS_H):
            b = perm[s:s + BS_H]
            opt.zero_grad()
            nn.functional.mse_loss(model(T[b], S[b], None if M is None else M[b]), Y[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = nn.functional.mse_loss(model(T[inner_va], S[inner_va],
                                              None if M is None else M[inner_va]), Y[inner_va]).item()
        if vl < best - 1e-5:
            best, bstate, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PAT_H:
                break
    model.load_state_dict(bstate); model.eval()
    with torch.no_grad():
        p = model(T[va], S[va], None if M is None else M[va]).cpu().numpy().squeeze(1) * ysd + ym
    d = p - yz[va]
    return (r2_score(yz[va], p), 1.4826 * np.median(np.abs(d - np.median(d))),
            float(np.mean(np.abs(d) > 0.5)))

print(f"pooling control, {CTRL_SPLITS} splits (the supervised head sees the mass labels)\n")
print(f"{'readout':38s} {'R2':>15s} {'nmad':>7s} {'out':>7s}")
for name, fn in [
    ("image: supervised attn-pool + linear",
     lambda tr, va: train_attnpool(gIMGP[zi], gISR[zi], None, tr, va)),
    ("image: CLIP aligned (ridge)",
     lambda tr, va: ridge_probe(IMG_AL, tr, va)),
    ("spectrum: supervised attn-pool + linear",
     lambda tr, va: train_attnpool(gSPEC[zi], gSSR[zi], gSM[zi], tr, va)),
    ("spectrum: CLIP aligned (ridge)",
     lambda tr, va: ridge_probe(SPC_AL, tr, va)),
]:
    R = np.array([fn(tr, va) for tr, va in splits[:CTRL_SPLITS]])
    print(f"{name:38s} {R[:,0].mean():7.3f}+/-{R[:,0].std():5.3f} {R[:,1].mean():7.3f} {R[:,2].mean():6.1%}")


### Pred-vs-true for every configuration

One fixed split (`PANEL_SEED`), one panel per feature set: baseline in the first column,
then unaligned (top row) against aligned (bottom row) for each modality and for the pair.


In [ ]:
# === pred-vs-true panels, one per configuration, on a single fixed split ===
PANEL_SEED = 0
tr_p, va_p = splits[PANEL_SEED]

def ridge_pred(X):
    sc = StandardScaler().fit(X[tr_p])
    return RidgeCV(alphas=ALPHAS).fit(sc.transform(X[tr_p]), yz[tr_p]).predict(sc.transform(X[va_p]))

panels = [("F150W photometry", PHOT),
          ("image\n(best unaligned)", IMG_BEST), ("image\n(aligned)", IMG_AL),
          ("spectrum\n(best unaligned)", SPC_CAT), ("spectrum\n(aligned)", SPC_AL),
          ("joint\n(best unaligned)", np.hstack([IMG_BEST, SPC_CAT])),
          ("joint\n(aligned)", gCLIP[zi])]

lo, hi = yz[va_p].min() - 0.3, yz[va_p].max() + 0.3
ticks  = [t for t in (8, 9, 10, 11, 12) if lo <= t <= hi]
ncol   = 4
nrow   = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.5 * nrow))
for ax, (name, X) in zip(axes.ravel(), panels):
    p = ridge_pred(X)
    t = yz[va_p]
    ax.scatter(t, p, s=10, alpha=0.3, color="#0072B2", edgecolors="none", rasterized=True)
    ax.plot([lo, hi], [lo, hi], "--", color="0.35", lw=1.2, alpha=0.75, dashes=(5, 3))
    d = p - t
    ax.text(0.96, 0.05,
            rf"$R^2$={r2_score(t, p):.3f}" + "\n"
            + rf"$\sigma_{{\rm NMAD}}$={1.4826 * np.median(np.abs(d - np.median(d))):.3f}" + "\n"
            + rf"$f_{{>0.5}}$={np.mean(np.abs(d) > 0.5):.1%}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
            bbox=dict(facecolor="white", alpha=0.7, pad=2.0, edgecolor="none"))
    ax.set_title(name, fontsize=10)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_box_aspect(1)
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_xlabel(r"catalog $\log M_\star$", fontsize=10)
    ax.set_ylabel(r"predicted $\log M_\star$", fontsize=10)
    for sp in ax.spines.values():
        sp.set_linewidth(1.4)
    ax.tick_params(which="both", direction="in", width=1.2, length=4,
                   top=True, bottom=True, left=True, right=True, labelsize=9)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
fig.patch.set_alpha(0.0)
fig.tight_layout()
outp = os.path.join(POSTER_DIR, "mass_1z3_configurations")
fig.savefig(outp + ".png", transparent=False, dpi=300, bbox_inches="tight")
print(f"Saved -> {outp}.png")
plt.show()
